In [10]:
# CLEANING AND MISSING DATA HANDLING
# Raw monitoring data is never clean. Metrics agents go offline — nulls appear. CSV exports from ServiceNow have inconsistent formatting. Timestamps arrive as strings. Duplicate rows appear when pipelines retry. A model trained on dirty data produces confidently wrong anomaly alerts.
# Garbage in — garbage out. Cleaning is not optional.

In [11]:
# LOAD DATASETS

import pandas as pd
df_metrics = pd.read_csv(
    "server_metrics.csv",
    parse_dates=["timestamp"],
    dtype={"server_id": "category"},
    na_values=["N/A", "null", "—"]
)

df_tickets = pd.read_csv(
    "incidents.csv",
    parse_dates=["created_at", "resolved_at"]
)

df_logs = pd.read_csv(
    "app_logs.csv",
    parse_dates=["timestamp"]
)

print(f"df_metrics : {df_metrics.shape}")
print(f"df_tickets : {df_tickets.shape}")
print(f"df_logs    : {df_logs.shape}")

df_metrics : (525, 7)
df_tickets : (213, 7)
df_logs    : (315, 5)


In [12]:
# DETECTING NULLS
print("Total nulls per columns:", df_metrics.isnull().sum())
print("percentage of nulls per columns:", df_metrics.isnull() / len(df_metrics) *100)
print("Any nulls at all", df_metrics.isnull().any())
print("Rows where any column has a null", df_metrics[df_metrics.isnull().any(axis=1)])

Total nulls per columns: timestamp      0
server_id      0
cpu_pct        0
memory_pct     0
response_ms    0
disk_pct       0
status         0
dtype: int64
percentage of nulls per columns:      timestamp  server_id  cpu_pct  memory_pct  response_ms  disk_pct  status
0          0.0        0.0      0.0         0.0          0.0       0.0     0.0
1          0.0        0.0      0.0         0.0          0.0       0.0     0.0
2          0.0        0.0      0.0         0.0          0.0       0.0     0.0
3          0.0        0.0      0.0         0.0          0.0       0.0     0.0
4          0.0        0.0      0.0         0.0          0.0       0.0     0.0
..         ...        ...      ...         ...          ...       ...     ...
520        0.0        0.0      0.0         0.0          0.0       0.0     0.0
521        0.0        0.0      0.0         0.0          0.0       0.0     0.0
522        0.0        0.0      0.0         0.0          0.0       0.0     0.0
523        0.0        0.0     

In [13]:
#DROPPING NULLS
print("Drop rows where any column has null", df_metrics.dropna())
print("Drop rows where specific column has null", df_metrics.dropna(subset=["cpu_pct"]))
print("Drop rows where ALL columns are null:", df_metrics.dropna(how="all"))
threshold = len(df_metrics) * 0.5
print("Drop columns with more than 50 % nulls", threshold)
df_metrics.dropna(axis=1, thresh=threshold)
#Use dropna() carefully in time series data — dropping rows breaks the time continuity. Filling is usually better than dropping for metric data.

Drop rows where any column has null               timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
0   2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
1   2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
2   2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
3   2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
4   2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
..                  ...       ...      ...         ...          ...       ...   
520 2026-01-01 04:00:00    srv-01     22.5        73.3        411.5      63.1   
521 2026-01-01 04:00:00    srv-02     39.7        57.9        918.9      44.9   
522 2026-01-01 04:00:00    srv-03     42.9        41.0       1119.2      82.5   
523 2026-01-01 04:00:00    srv-04     88.8        84.6        264.6      88.0   
524 2026-01-01 04:00:00    srv-05     83.8        90.9        415.7      

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status
0,2026-01-01 00:00:00,srv-01,49.6,94.6,891.8,68.9,ok
1,2026-01-01 00:00:00,srv-02,32.3,33.9,1046.1,69.1,ok
2,2026-01-01 00:00:00,srv-03,21.6,96.0,1007.3,43.8,ok
3,2026-01-01 00:00:00,srv-04,34.5,50.7,653.5,58.1,ok
4,2026-01-01 00:00:00,srv-05,68.3,39.5,386.0,53.8,ok
...,...,...,...,...,...,...,...
520,2026-01-01 04:00:00,srv-01,22.5,73.3,411.5,63.1,warning
521,2026-01-01 04:00:00,srv-02,39.7,57.9,918.9,44.9,ok
522,2026-01-01 04:00:00,srv-03,42.9,41.0,1119.2,82.5,ok
523,2026-01-01 04:00:00,srv-04,88.8,84.6,264.6,88.0,ok


In [14]:
# Filling nulls - fillna()
print("Fill with a fixed value", df_metrics["cpu_pct"].fillna(0))
print("Fill with column mean - good for metric data", df_metrics["cpu_pct"].fillna(df_metrics["cpu_pct"].mean()))
print("Fill with column median - better when outliers exist", df_metrics["cpu_pct"].fillna(df_metrics["cpu_pct"].median()))
print("Fill with forward fill - use previous valid value", df_metrics["cpu_pct"].fillna(method="ffill"))
print("Fill with backward fill - use next valid value", df_metrics["cpu_pct"].fillna(method="bfill"))

Fill with a fixed value 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22.5
521    39.7
522    42.9
523    88.8
524    83.8
Name: cpu_pct, Length: 525, dtype: float64
Fill with column mean - good for metric data 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22.5
521    39.7
522    42.9
523    88.8
524    83.8
Name: cpu_pct, Length: 525, dtype: float64
Fill with column median - better when outliers exist 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22.5
521    39.7
522    42.9
523    88.8
524    83.8
Name: cpu_pct, Length: 525, dtype: float64
Fill with forward fill - use previous valid value 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22.5
521    39.7
522    42.9
523    88.8
524    83.8
Name: cpu_pct, Length: 525, dtype: float64
Fill with backward fill - use next valid value 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22

C:\Users\HOME\AppData\Local\Temp\ipykernel_5352\1478460071.py:5: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  print("Fill with forward fill - use previous valid value", df_metrics["cpu_pct"].fillna(method="ffill"))
C:\Users\HOME\AppData\Local\Temp\ipykernel_5352\1478460071.py:6: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  print("Fill with backward fill - use next valid value", df_metrics["cpu_pct"].fillna(method="bfill"))


In [15]:
# interpolate() - smooth gap filling
# Better than forward fill for gradual metric changes
print("Linear interpolation - fills gap with straight line between known values", df_metrics["cpu_pct"].interpolate(method="linear"))
print("Time-based interpolation - accounts for unevan time gaps")
df_metrics.set_index("timestamp")["cpu_pct"].interpolate(method="time")

Linear interpolation - fills gap with straight line between known values 0      49.6
1      32.3
2      21.6
3      34.5
4      68.3
       ... 
520    22.5
521    39.7
522    42.9
523    88.8
524    83.8
Name: cpu_pct, Length: 525, dtype: float64
Time-based interpolation - accounts for unevan time gaps


timestamp
2026-01-01 00:00:00    49.6
2026-01-01 00:00:00    32.3
2026-01-01 00:00:00    21.6
2026-01-01 00:00:00    34.5
2026-01-01 00:00:00    68.3
                       ... 
2026-01-01 04:00:00    22.5
2026-01-01 04:00:00    39.7
2026-01-01 04:00:00    42.9
2026-01-01 04:00:00    88.8
2026-01-01 04:00:00    83.8
Name: cpu_pct, Length: 525, dtype: float64

In [16]:
# Detecting and removing duplicates
print("Check for duplicate rows:", df_metrics.duplicated().sum())

Check for duplicate rows: 25


In [17]:
print("see which rows are duplicated", df_metrics[df_metrics.duplicated()])

see which rows are duplicated               timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
500 2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
501 2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
502 2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
503 2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
504 2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
505 2026-01-01 01:00:00    srv-01     82.0        43.6        641.4      68.5   
506 2026-01-01 01:00:00    srv-02     68.0        41.6        124.8      91.7   
507 2026-01-01 01:00:00    srv-03     83.9        50.7        162.3      74.5   
508 2026-01-01 01:00:00    srv-04     29.6        63.7         89.5      89.1   
509 2026-01-01 01:00:00    srv-05     72.3        51.2        648.1      65.5   
510 2026-01-01 02:00:00    srv-01     96.6        82.7       1130.4      88.2  

In [28]:
print("Duplicates based on specific columns", df_metrics.duplicated(subset=["server_id", "timestamp"]).sum())

Duplicates based on specific columns 25


In [29]:
print("Remove duplicates - keep first occurence:", df_metrics.drop_duplicates())

Remove duplicates - keep first occurence:               timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
0   2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
1   2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
2   2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
3   2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
4   2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
..                  ...       ...      ...         ...          ...       ...   
495 2026-01-05 03:00:00    srv-01     88.7        38.9        959.1      38.1   
496 2026-01-05 03:00:00    srv-02     41.8        89.6       1135.6      39.7   
497 2026-01-05 03:00:00    srv-03     97.5        62.9       1043.1      68.3   
498 2026-01-05 03:00:00    srv-04     42.6        43.8        926.1      55.1   
499 2026-01-05 03:00:00    srv-05     58.9        69.3       1045.4

In [31]:
print("Remove duplicates on specific columns", df_metrics.drop_duplicates(subset=["server_id", "timestamp"]))

Remove duplicates on specific columns               timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
0   2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
1   2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
2   2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
3   2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
4   2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   
..                  ...       ...      ...         ...          ...       ...   
495 2026-01-05 03:00:00    srv-01     88.7        38.9        959.1      38.1   
496 2026-01-05 03:00:00    srv-02     41.8        89.6       1135.6      39.7   
497 2026-01-05 03:00:00    srv-03     97.5        62.9       1043.1      68.3   
498 2026-01-05 03:00:00    srv-04     42.6        43.8        926.1      55.1   
499 2026-01-05 03:00:00    srv-05     58.9        69.3       1045.4    

In [32]:
# Fixing dytpes
print("Check all dtypes:", df_metrics.dtypes)

Check all dtypes: timestamp      datetime64[ns]
server_id            category
cpu_pct               float64
memory_pct            float64
response_ms           float64
disk_pct              float64
status                 object
dtype: object


In [38]:
# string ----> datetime
df_metrics["timestamp"] = pd.to_datetime(df_metrics["timestamp"])

In [39]:
# String ---> numeric
df_metrics["cpu_pct"] = pd.to_numeric(df_metrics["cpu_pct"], errors="coerce")
# errors="coerce" → converts bad values to NaN instead of throwing error

In [40]:
# Numeric ---> category - for low-cardinality string columns
df_metrics["status"] = df_metrics["status"].astype("category")
df_metrics["server_id"] = df_metrics["server_id"].astype("category")

In [42]:
# String ---> boolean
df_tickets["resolved"] = df_tickets["resolved"].astype(bool)

In [ ]:
###################Cleaning string columns################

In [43]:
print("Inconsistent category values from servicenow exports", df_tickets["category"].unique())

Inconsistent category values from servicenow exports ['Service Down' 'CPU High' 'Disk Full' 'Network Lag' 'Memory Leak']


In [44]:
# standardize - strip whitespace and lowercase
df_tickets["category"] = df_tickets["category"].str.strip().str.lower()

In [45]:
# Replacing inconsistent values
df_tickets["category"] = df_tickets["category"].replace({
    "cpu hight": "CPU High",
    "disk full": "Disk Full",
    "network lag": "Network Lag"
})

In [ ]:
#####################################   Handling outliers - Identity and flag ###########################

In [46]:
# Z-score method - flag values beyond 3 standar deviations
mean = df_metrics["response_ms"].mean()
std = df_metrics["response_ms"].std()
df_metrics["response_ms_zscore"] = (df_metrics["response_ms"] - mean) / std
df_metrics["is_outlier"] = df_metrics["response_ms_zscore"].abs() > 3
print(df_metrics["is_outlier"].sum())

0


In [47]:
# IQR method — more robust than z-score for skewed distributions
Q1  = df_metrics["response_ms"].quantile(0.25)
Q3  = df_metrics["response_ms"].quantile(0.75)
IQR = Q3 - Q1

outliers = df_metrics[
    (df_metrics["response_ms"] < Q1 - 1.5 * IQR) |
    (df_metrics["response_ms"] > Q3 + 1.5 * IQR)
]
print(f"Outliers detected : {len(outliers)}")

 # outlier response times are not errors to remove. They are incidents to investigate. Flag them, don't drop them.

Outliers detected : 0
